# SafeRunner integration

Wrap a plain function (or an object with `run(dict)`) with the same **RAI policy** as the main executor. Use `include_trace=True` for a `PipelineResult` compatible with `to_run_summary()`.

In [1]:
from oris.core.exceptions import GuardViolationError
from oris.integrations import SafeRunner
from oris.rai.policy import PolicyEnforcer

policy = PolicyEnforcer()


def my_app(data: dict) -> dict:
    q = data.get("query", "")
    return {"answer": f"Processed: {q!r}", "ok": True}


runner = SafeRunner(my_app, policy=policy)
out = runner.run({"query": "Hello from SafeRunner"})
out

{'answer': "Processed: 'Hello from SafeRunner'", 'ok': True}

In [2]:
traced = runner.run({"query": "Trace me"}, include_trace=True)
summ = traced.to_run_summary()
summ["status"], summ["trace"][0]["step_id"], summ["trace"][0]["flags"]

('success', 'external_pipeline', {'kind': 'external_pipeline'})

In [3]:
try:
    runner.run({"secret": "not-allowed-key"})
except GuardViolationError as exc:
    print("Blocked by input policy:", exc)

Blocked by input policy: Input contains prohibited key 'secret'.


Consistent **dict** output on success; **`GuardViolationError`** when input or output policy rejects the payload.